# 🌍 FirstVoice: Offline Multilingual Crisis Communication Agent**Powered by Gemma 4 E4B** | Kaggle Gemma 4 Good Hackathon> When disasters strike, communication collapses. FirstVoice turns any phone into an AI bridge between responders and survivors — completely offline.This notebook demonstrates:1. 🎤 Audio Transcription (native audio encoder)2. 🌐 Language Detection + Translation3. 📸 Vision Analysis (damage assessment)4. 📋 Triage Card (native function calling)5. 💬 Quick Phrases6. 🖥️ Gradio Demo UI

## ⚙️ Setup

In [ ]:
!pip install -q "git+https://github.com/huggingface/transformers.git" accelerate torch gradio scipy soundfile Pillow

In [ ]:
import torchfrom transformers import AutoProcessor, AutoModelForMultimodalLMimport json, reMODEL_ID = "google/gemma-4-E4B-it"print("Loading Gemma 4 E4B...")processor = AutoProcessor.from_pretrained(MODEL_ID)model = AutoModelForMultimodalLM.from_pretrained(    MODEL_ID, torch_dtype=torch.float16, device_map="auto")print(f"Model loaded on {model.device}")

## 🔧 Helper Functions

In [ ]:
def generate(messages, max_tokens=512, tools=None):    text = processor.apply_chat_template(        messages, tools=tools, tokenize=False, add_generation_prompt=True    )    inputs = processor(text=text, return_tensors="pt").to(model.device)    outputs = model.generate(**inputs, max_new_tokens=max_tokens)    generated = outputs[0][len(inputs["input_ids"][0]):]    return processor.decode(generated, skip_special_tokens=True)def generate_multimodal(messages, max_tokens=512):    inputs = processor.apply_chat_template(        messages, add_generation_prompt=True,        tokenize=True, return_dict=True, return_tensors="pt"    )    inputs = inputs.to(model.device, dtype=model.dtype)    outputs = model.generate(**inputs, max_new_tokens=max_tokens)    text = processor.batch_decode(outputs, skip_special_tokens=True)[0]    parts = text.split("model\n")    return parts[-1].strip() if len(parts) > 1 else text.strip()

## 🎤 Demo 1: Audio TranscriptionGemma 4 E4B has a native audio encoder (300M params). Direct speech-to-text, no separate ASR model.

In [ ]:
def transcribe_audio(audio_path):    messages = [{        "role": "user",        "content": [            {"type": "text", "text": "Transcribe the following speech segment in its original language. Only output the transcription with no newlines. Before the transcription, write LANGUAGE: followed by the detected language name."},            {"type": "audio", "audio": audio_path},        ]    }]    return generate_multimodal(messages, max_tokens=256)print("Audio transcription ready.")print("Usage: transcribe_audio('path/to/file.wav')")

## 🌐 Demo 2: Language Detection + TranslationGemma 4 supports 140+ languages. Single call detects and translates.

In [ ]:
def detect_and_translate(text, target_language="English"):    messages = [{        "role": "user",        "content": f"Detect the language of this text and translate it to {target_language}.\nRespond in this exact format (2 lines only):\nLANG: <detected language>\nTEXT: <translated text>\n\n\"{text}\""    }]    result = generate(messages, max_tokens=256)    lang, translated = "Unknown", result    for line in result.strip().split("\n"):        if line.startswith("LANG:"): lang = line[5:].strip()        elif line.startswith("TEXT:"): translated = line[5:].strip()    return {"detected_language": lang, "translated_text": translated}print("--- Hindi ---")print(detect_and_translate("मुझे पानी चाहिए, मेरा बच्चा बीमार है"))print("\n--- Bengali ---")print(detect_and_translate("আমি আটকে আছি, সাহায্য করুন"))print("\n--- Arabic ---")print(detect_and_translate("أنا محاصر تحت المبنى"))print("\n--- Spanish ---")print(detect_and_translate("Necesito ayuda médica urgente"))

## 📸 Demo 3: Vision AnalysisGemma 4 vision encoder analyzes disaster photos for damage, injuries, and hazards.

In [ ]:
def analyze_damage(image_path):    messages = [{        "role": "user",        "content": [            {"type": "image", "image": image_path},            {"type": "text", "text": "You are a disaster damage assessment AI. Analyze this image.\nRespond in JSON:\n{\n  \"structural_severity\": \"NONE/MINOR/MODERATE/SEVERE/CATASTROPHIC\",\n  \"structural_description\": \"...\",\n  \"hazards\": [{\"type\": \"...\", \"severity\": \"...\"}],\n  \"summary\": \"one sentence\"\n}"}        ]    }]    return generate_multimodal(messages, max_tokens=512)print("Vision analysis ready.")print("Usage: analyze_damage('path/to/image.jpg')")

## 📋 Demo 4: Triage Card Generation (Function Calling)Gemma 4 native function calling generates structured triage cards.

In [ ]:
triage_tool = {    "type": "function",    "function": {        "name": "generate_triage_card",        "description": "Generate a structured triage card from a disaster encounter.",        "parameters": {            "type": "object",            "properties": {                "people_count": {"type": "integer", "description": "Number of people affected"},                "urgency_level": {"type": "string", "description": "CRITICAL, HIGH, MEDIUM, or LOW"},                "needs_categories": {"type": "string", "description": "Comma-separated: Medical, Extraction, Shelter, WaterFood, FamilyReunification"},                "detected_language": {"type": "string", "description": "Language the survivor spoke"},                "assessment_summary": {"type": "string", "description": "Concise situation summary"}            },            "required": ["urgency_level", "needs_categories", "detected_language", "assessment_summary"]        }    }}def generate_triage_card(conversation):    messages = [        {"role": "system", "content": "You are a disaster triage agent. Analyze the encounter and call generate_triage_card."},        {"role": "user", "content": f"Analyze this encounter and generate a triage card:\n\n{conversation}"}    ]    return generate(messages, max_tokens=256, tools=[triage_tool])sample = """[Survivor] (Hindi): मुझे पानी चाहिए, मेरा बच्चा बीमार है  Translated: I need water, my child is sick[Responder] (English): How many people are with you?[Survivor] (Hindi): हम पांच लोग हैं, एक बुजुर्ग महिला घायल है  Translated: We are 5 people, one elderly woman is injured[Photo Assessment]: MODERATE structural damage, partial roof collapse"""print("Generating triage card...")print(generate_triage_card(sample))

## 💬 Demo 5: Quick PhrasesPre-translated emergency phrases — instant communication without AI delay.

In [ ]:
QUICK_PHRASES = {    "medical": ["Are you injured?", "Where does it hurt?", "Can you move your legs?", "Do you have any allergies?"],    "safety": ["Help is coming.", "Stay calm.", "Do not move.", "We need to evacuate now."],    "logistics": ["How many people are with you?", "Is anyone trapped?", "Do you need water?", "What is your name?"]}def translate_phrase(phrase, target):    messages = [{"role": "user", "content": f"Translate to {target}. Output ONLY the translation:\n\"{phrase}\""}]    return generate(messages, max_tokens=64).strip().strip('"')print("=== Emergency Phrases -> Hindi ===")for cat, phrases in QUICK_PHRASES.items():    print(f"\n{cat.upper()}")    for p in phrases[:2]:        print(f"  EN: {p}")        print(f"  HI: {translate_phrase(p, 'Hindi')}")

## 🖥️ Interactive Demo (Gradio)

In [ ]:
import gradio as grdef ui_translate(text, target_lang):    if not text.strip(): return "Please enter text."    r = detect_and_translate(text, target_lang)    return f"Detected: {r['detected_language']}\nTranslation: {r['translated_text']}"def ui_triage(conversation):    if not conversation.strip(): return "Please enter a conversation."    return generate_triage_card(conversation)def ui_transcribe(audio):    if audio is None: return "Please upload a WAV file."    return transcribe_audio(audio)def ui_vision(image):    if image is None: return "Please upload an image."    return analyze_damage(image)with gr.Blocks(title="FirstVoice") as demo:    gr.Markdown("# 🌍 FirstVoice: Offline Crisis Communication\n*Powered by Gemma 4 E4B*")    with gr.Tab("🌐 Translation"):        with gr.Row():            txt_in = gr.Textbox(label="Input (any language)", lines=3)            lang_sel = gr.Dropdown(["English","Hindi","Bengali","Tamil","Arabic","Spanish","French"], value="English", label="Target")        txt_out = gr.Textbox(label="Result", lines=3)        gr.Button("Translate").click(ui_translate, [txt_in, lang_sel], txt_out)    with gr.Tab("🎤 Audio"):        audio_in = gr.Audio(type="filepath", label="Upload WAV")        audio_out = gr.Textbox(label="Transcription", lines=3)        gr.Button("Transcribe").click(ui_transcribe, audio_in, audio_out)    with gr.Tab("📸 Vision"):        img_in = gr.Image(type="filepath", label="Upload Photo")        img_out = gr.Textbox(label="Assessment", lines=6)        gr.Button("Analyze").click(ui_vision, img_in, img_out)    with gr.Tab("📋 Triage"):        conv_in = gr.Textbox(label="Conversation", lines=6)        triage_out = gr.Textbox(label="Triage Card", lines=5)        gr.Button("Generate").click(ui_triage, conv_in, triage_out)demo.launch(share=True)